<a href="https://colab.research.google.com/github/hye0-n0/AI_finance/blob/main/%5B0720%5DLSTM_2000model_validfront.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import os
import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import repeat
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow.keras as keras
from keras.models import Sequential
from keras.layers import Dense, LSTM, GRU, RNN, Dropout, Flatten
from keras.optimizers import SGD
from sklearn.impute import SimpleImputer
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
import torch
import torch.nn as nn
from torchsummary import summary


import warnings
warnings.filterwarnings("ignore")

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42) # Seed 고정

## DataLoad

공유드라이브는 mounting이 힘들기 때문에 </br>공유함에 있는 데이터 파일 open의 shortcut을 nyDrive에 만들면 됩니다

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_i = pd.read_csv('/content/drive/MyDrive/open/train.csv')
df_i

,일자,종목코드,종목명,거래량,시가,고가,저가,종가
0,20210601,A060310,3S,166690,2890,2970,2885,2920
1,20210601,A095570,AJ네트웍스,63836,5860,5940,5750,5780
2,20210601,A006840,AK홀딩스,103691,35500,35600,34150,34400
3,20210601,A054620,APS,462544,14600,14950,13800,14950
4,20210601,A265520,AP시스템,131987,29150,29150,28800,29050
...,...,...,...,...,...,...,...,...
987995,20230530,A189980,흥국에프엔비,272284,3005,3035,2955,2980
987996,20230530,A000540,흥국화재,50218,3250,3255,3195,3215
987997,20230530,A003280,흥아해운,130664,1344,1395,1340,1370
987998,20230530,A037440,희림,141932,9170,9260,9170,9200


In [ ]:
# 그냥 불러와서 사용하는 것도 GOOD choice
df = pd.read_csv('/content/drive/MyDrive/busy_kids/dacon_krx/data/stock_data_with_interest_rate.csv', dtype={'StockCode': str})
df

,Date,StockCode,Open,High,Low,Close,Volume,Change,금리
0,2020-05-29,000020,10900,12150,10850,11550,4470917,0.069444,0.5
1,2020-06-01,000020,11750,11900,11450,11600,1547269,0.004329,0.5
2,2020-06-02,000020,11600,11800,10950,11750,1171851,0.012931,0.5
3,2020-06-03,000020,12100,12100,11450,11600,1193172,-0.012766,0.5
4,2020-06-04,000020,11850,11850,11250,11350,975698,-0.021552,0.5
...,...,...,...,...,...,...,...,...,...
1473725,2023-05-23,383800,8390,8390,8310,8330,150364,-0.003589,3.5
1473726,2023-05-24,383800,8310,8340,8280,8300,122457,-0.003601,3.5
1473727,2023-05-25,383800,8300,8310,8270,8310,84241,0.001205,3.5
1473728,2023-05-26,383800,8300,8310,8270,8280,126681,-0.003610,3.5


##(참고용)Data Windows
* 예측 시점 하루 -> 15일 예측
* target에 따라 두가지 버전


In [ ]:
def get_target_for_single_stock(data):
    df = data.copy()
    df['target'] = df['Close'].shift(-15).fillna(method='ffill')  # NaN 값을 먼저 채웁니다.
    df['target'] = df['target'].astype(int)/1000  # 소수점을 제거합니다.

    return df

def get_target_for_all_stocks(data):
    # 데이터 프레임을 종목별로 분리합니다.
    grouped_data = data.groupby('StockCode')

    # 각 종목별로 `get_target_for_single_stock` 함수를 적용합니다.
    dfs_with_target = [get_target_for_single_stock(group) for _, group in grouped_data]

    # 처리된 데이터 프레임들을 하나로 합칩니다.
    combined_data = pd.concat(dfs_with_target, ignore_index=True)
    return combined_data

In [ ]:
data_with_target = get_target_for_all_stocks(df)

data_with_target


,Date,StockCode,Open,High,Low,Close,Volume,Change,금리,target
0,2020-05-29,000020,10900,12150,10850,11550,4470917,0.069444,0.5,15.70
1,2020-06-01,000020,11750,11900,11450,11600,1547269,0.004329,0.5,16.00
2,2020-06-02,000020,11600,11800,10950,11750,1171851,0.012931,0.5,16.85
3,2020-06-03,000020,12100,12100,11450,11600,1193172,-0.012766,0.5,16.90
4,2020-06-04,000020,11850,11850,11250,11350,975698,-0.021552,0.5,17.50
...,...,...,...,...,...,...,...,...,...,...
1473725,2023-05-23,383800,8390,8390,8310,8330,150364,-0.003589,3.5,8.29
1473726,2023-05-24,383800,8310,8340,8280,8300,122457,-0.003601,3.5,8.29
1473727,2023-05-25,383800,8300,8310,8270,8310,84241,0.001205,3.5,8.29
1473728,2023-05-26,383800,8300,8310,8270,8280,126681,-0.003610,3.5,8.29


In [ ]:
df_for_scale = data_with_target.copy()

# 우선, 스케일링을 진행할 열만 선택합니다.
columns_to_scale = ['Open', 'High', 'Low', 'Close', 'Volume']
data_to_scale = df[columns_to_scale]

# 스케일링을 진행합니다.
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data_to_scale)


# 칼럼을 스케일링하고 원래 데이터프레임에서 대체
scaled_columns = scaler.fit_transform(df_for_scale[columns_to_scale])  # 스케일링된 칼럼 얻기
df_for_scale[columns_to_scale] = scaled_columns  # 스케일링된 칼럼으로 대체

df_for_scale

,Date,StockCode,Open,High,Low,Close,Volume,Change,금리,target
0,2020-05-29,000020,0.006158,0.006811,0.006179,0.006446,0.003720,0.069444,0.5,15.70
1,2020-06-01,000020,0.006638,0.006670,0.006521,0.006474,0.001287,0.004329,0.5,16.00
2,2020-06-02,000020,0.006554,0.006614,0.006236,0.006559,0.000975,0.012931,0.5,16.85
3,2020-06-03,000020,0.006836,0.006783,0.006521,0.006474,0.000993,-0.012766,0.5,16.90
4,2020-06-04,000020,0.006695,0.006642,0.006407,0.006333,0.000812,-0.021552,0.5,17.50
...,...,...,...,...,...,...,...,...,...,...
1473725,2023-05-23,383800,0.004740,0.004703,0.004732,0.004626,0.000125,-0.003589,3.5,8.29
1473726,2023-05-24,383800,0.004695,0.004675,0.004715,0.004609,0.000102,-0.003601,3.5,8.29
1473727,2023-05-25,383800,0.004689,0.004658,0.004710,0.004615,0.000070,0.001205,3.5,8.29
1473728,2023-05-26,383800,0.004689,0.004658,0.004710,0.004598,0.000105,-0.003610,3.5,8.29


In [ ]:
df_for_scale['Date'] = pd.to_datetime(df_for_scale['Date'], format='%Y-%m-%d')
# valid:train = 2:8 = 145:582(일) => 2020-12-24 기준으로 나눔
# test: '2023-05-09', '2023-05-30'(15day) -> '2023-06-'
valid_df_for_scale = df_for_scale[df_for_scale['Date'].between('2020-05-29', '2020-12-24')]
valid_df_for_scale

,Date,StockCode,Open,High,Low,Close,Volume,Change,금리,target
0,2020-05-29,000020,0.006158,0.006811,0.006179,0.006446,0.003720,0.069444,0.5,15.700
1,2020-06-01,000020,0.006638,0.006670,0.006521,0.006474,0.001287,0.004329,0.5,16.000
2,2020-06-02,000020,0.006554,0.006614,0.006236,0.006559,0.000975,0.012931,0.5,16.850
3,2020-06-03,000020,0.006836,0.006783,0.006521,0.006474,0.000993,-0.012766,0.5,16.900
4,2020-06-04,000020,0.006695,0.006642,0.006407,0.006333,0.000812,-0.021552,0.5,17.500
...,...,...,...,...,...,...,...,...,...,...
1468767,2020-12-22,365590,0.001101,0.001106,0.001107,0.001029,0.000083,0.007696,0.5,1.983
1468768,2020-12-23,365590,0.001109,0.001117,0.001113,0.001037,0.000042,0.007128,0.5,1.978
1468769,2020-12-24,365590,0.001118,0.001117,0.001124,0.001037,0.000012,0.000000,0.5,1.978
1469950,2020-12-23,369370,0.001840,0.001930,0.001855,0.001788,0.000476,NaN,0.5,3.346


In [ ]:
train_df_for_scale = df_for_scale[df_for_scale['Date'].between('2020-12-25', '2023-05-08')]
train_df_for_scale

,Date,StockCode,Open,High,Low,Close,Volume,Change,금리,target
145,2020-12-28,000020,0.011299,0.011239,0.010592,0.010457,0.001011,-0.067500,0.5,16.90
146,2020-12-29,000020,0.010593,0.010874,0.010678,0.010740,0.000392,0.026810,0.5,16.70
147,2020-12-30,000020,0.010791,0.011099,0.010706,0.011022,0.000519,0.026110,0.5,16.40
148,2021-01-04,000020,0.011158,0.011155,0.010735,0.010712,0.000598,-0.027990,0.5,17.00
149,2021-01-05,000020,0.010734,0.010930,0.010621,0.010881,0.000550,0.015707,0.5,16.70
...,...,...,...,...,...,...,...,...,...,...
1473710,2023-04-28,383800,0.004842,0.004809,0.004841,0.004734,0.000113,-0.004673,3.5,8.33
1473711,2023-05-02,383800,0.004819,0.004804,0.004852,0.004745,0.000056,0.002347,3.5,8.30
1473712,2023-05-03,383800,0.004825,0.004832,0.004852,0.004756,0.000110,0.002342,3.5,8.31
1473713,2023-05-04,383800,0.004836,0.004832,0.004875,0.004773,0.000059,0.003505,3.5,8.28


In [ ]:
pred_df_for_scale = df_for_scale[df_for_scale['Date'].between('2023-05-09', '2023-05-30')]
pred_df_for_scale

,Date,StockCode,Open,High,Low,Close,Volume,Change,금리,target
728,2023-05-09,000020,0.004859,0.004882,0.004852,0.004785,0.000065,0.003497,3.5,9.70
729,2023-05-10,000020,0.004870,0.004849,0.004863,0.004779,0.000022,-0.001161,3.5,9.70
730,2023-05-11,000020,0.004859,0.004843,0.004841,0.004722,0.000045,-0.011628,3.5,9.70
731,2023-05-12,000020,0.004802,0.004787,0.004767,0.004700,0.000042,-0.004706,3.5,9.70
732,2023-05-15,000020,0.004751,0.004759,0.004755,0.004694,0.000029,-0.001182,3.5,9.70
...,...,...,...,...,...,...,...,...,...,...
1473725,2023-05-23,383800,0.004740,0.004703,0.004732,0.004626,0.000125,-0.003589,3.5,8.29
1473726,2023-05-24,383800,0.004695,0.004675,0.004715,0.004609,0.000102,-0.003601,3.5,8.29
1473727,2023-05-25,383800,0.004689,0.004658,0.004710,0.004615,0.000070,0.001205,3.5,8.29
1473728,2023-05-26,383800,0.004689,0.004658,0.004710,0.004598,0.000105,-0.003610,3.5,8.29


In [ ]:
def window_dataset(data, time_steps=20, for_periods=1):
    grouped_data = data.groupby('StockCode')  # 데이터 프레임을 종목별로 분리합니다.
    all_stock_data = []

    for stock_code, sample in tqdm(grouped_data):
        # 데이터 변환
        ts_data = sample.values
        ts_data_len = len(ts_data)
        start_index = sample.columns.to_list().index('Open')
        end_index = sample.columns.to_list().index('금리') + 1
        close_index = sample.columns.to_list().index('target')  # 종가: target -> Close

        # 입력 데이터와 타겟 데이터 생성
        inout_seq = []
        X_data = []
        y_data = []
        for i in range(0, ts_data_len - time_steps - for_periods + 1):
            X_data.append(ts_data[i : i+time_steps, start_index:end_index].astype(np.float32))
            y_data.append(ts_data[i+time_steps : i+time_steps+for_periods, close_index].astype(np.float32))

            inout_seq.append((ts_data[i : i+time_steps, start_index:end_index].astype(np.float32), ts_data[i+time_steps : i+time_steps+for_periods, close_index].astype(np.float32)))

        X_data, y_data = np.array(X_data), np.array(y_data)
        all_stock_data.append((stock_code, X_data, y_data, inout_seq))

    return all_stock_data


In [ ]:
result = window_dataset(train_df_for_scale)

  0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
print(len(result[0][3]))

563


In [ ]:
print(len(result))
result[0][1].shape[2]

2000


7

##Model Define, Train and Inference


###(참고용) LSTM

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1)
if device == 'cuda':
    torch.cuda.manual_seed_all(1)

In [ ]:
# Define the RNN model
class myLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(myLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bias = True, dropout = 0.25)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.fc = nn.Linear(64, 1)
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.shape[0], self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.shape[0], self.hidden_size).to(device)
        output, (hn, cn) = self.lstm(x, (h0, c0))
        h_out = hn.view(-1, self.hidden_size) # h_out을 출력으로 사용한다.
        out = self.fc1(h_out)
        out = self.tanh(out)
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [ ]:
# Hyperparameters
input_size = result[0][1].shape[2]
hidden_size = 128
num_layers = 1
learning_rate = 0.005
batch_size = 128
num_epochs = 1000
patience = 10
verbose = 20

In [ ]:
def train_and_predict(result, pred, batch_size, learning_rate, hidden_size, num_layers, num_epochs, patience, input_size=7, verbose=1, device='cuda'):


    trained_models = {}
    predictions = []

    for stock_data in tqdm(result):
        stock_code, X_data, y_data, inout_seq = stock_data

        # Preparing the dataset
        class TrainDataset(Dataset):
          def __init__(self, X_data, y_data):
            self.X_data = X_data
            self.y_data = y_data

          def __len__(self):
            return len(self.y_data)

          def __getitem__(self, idx):
            return self.X_data[idx], self.y_data[idx]
        train_data = TrainDataset(X_data, y_data)
        train_loader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=False)

        #predict data 5.08~5.30
        pred_df = pred_df_for_scale[pred_df_for_scale['StockCode'] == stock_code]
        pred_df = pred_df.values
        X_pred = [pred_df[:, 2:-1]]
        X_pred = Variable(torch.Tensor(X_pred))  # ~5/30까지 15일

        #model
        model = myLSTM(input_size, hidden_size, num_layers).to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        optimizer.zero_grad()

        train_hist = np.zeros(num_epochs)
        print(f"Training for stock {stock_code}")

        for epoch in range(num_epochs):
            avg_cost = 0
            total_batch = len(train_loader)

            for inputs, labels in train_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs.float(), labels.float())
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                avg_cost += loss / total_batch

            train_hist[epoch] = avg_cost
            if epoch % verbose == 0:
                print('Epoch:', '%04d' % (epoch), 'train loss :', '{:.4f}'.format(avg_cost))

            if (epoch % patience == 0) & (epoch != 0):
                if train_hist[epoch - patience] < train_hist[epoch]:
                    print('\nEarly Stopping')
                    break

        print(f"Finished training for stock {stock_code}")
        trained_models[stock_code] = model.eval()

        # 예측 값 계산
        prediction = model(torch.tensor(X_pred).float().to(device)).item()
        predictions.append((stock_code, prediction))

        # 모델 제거
        del model
        del trained_models[stock_code]

    # 결과 표시
    prediction_df = pd.DataFrame(predictions, columns=["StockCode", "final_return"])
    print(prediction_df)
    return prediction_df

# 함수 호출 예제
prediction_result = train_and_predict(result, pred_df_for_scale, batch_size=256, learning_rate=0.5, hidden_size=15, num_layers=1, num_epochs=100, patience=10, input_size=7, verbose=20, device='cuda')


  0%|          | 0/2000 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
Epoch: 0040 train loss : 74.2237

Early Stopping
Finished training for stock 092230
Training for stock 092300
Epoch: 0000 train loss : 35.0885
Epoch: 0020 train loss : 0.7049

Early Stopping
Finished training for stock 092300
Training for stock 092440
Epoch: 0000 train loss : 39.5277
Epoch: 0020 train loss : 1.0647
Epoch: 0040 train loss : 0.7357

Early Stopping
Finished training for stock 092440
Training for stock 092460
Epoch: 0000 train loss : 43.1635
Epoch: 0020 train loss : 1.3733
Epoch: 0040 train loss : 0.8358

Early Stopping
Finished training for stock 092460
Training for stock 092730
Epoch: 0000 train loss : 418.4457
Epoch: 0020 train loss : 41.4005

Early Stopping
Finished training for stock 092730
Training for stock 092780
Epoch: 0000 train loss : 42.2951
Epoch: 0020 train loss : 1.9169

Early Stopping
Finished training for stock 092780
Training for stock 092870
Epoch: 0000 train loss : 98.2053
Epoch: 0020 train loss : 8.119

In [ ]:
prediction_result.tail(500)
prediction_result.to_csv('./prediction_result.csv', index=False)

In [ ]:
last_rows = df.groupby('StockCode').last().reset_index()
last_rows

,StockCode,Date,Open,High,Low,Close,Volume,Change,금리
0,000020,2023-05-30,9960,10040,9640,9700,201361,-0.015228,3.5
1,000040,2023-05-30,704,707,676,680,430077,-0.028571,3.5
2,000050,2023-05-30,10430,10480,10350,10440,7087,0.000000,3.5
3,000070,2023-05-30,72900,74000,72000,72600,4051,-0.001376,3.5
4,000080,2023-05-30,23700,23750,22900,23000,412668,-0.019190,3.5
...,...,...,...,...,...,...,...,...,...
1995,375500,2023-05-30,36850,37200,36200,36750,170274,0.023677,3.5
1996,378850,2023-05-30,4520,4580,4350,4460,88590,-0.004464,3.5
1997,383220,2023-05-30,135000,135200,131100,131700,151534,-0.022272,3.5
1998,383310,2023-05-30,62600,62800,61200,62000,116680,-0.001610,3.5


In [ ]:
last_close_df = last_rows[['StockCode', 'Close']]
merged_df = prediction_result.merge(last_close_df, on='StockCode')
merged_df['changes'] = (merged_df['final_return'] - merged_df['Close']) / merged_df['Close']

prediction_result['changes'] = merged_df['changes']

In [ ]:
prediction_result.head(200)

,StockCode,final_return,changes
0,000020,9.615606,-0.999009
1,000040,0.703073,-0.998966
2,000050,12.264784,-0.998825
3,000070,83.353607,-0.998852
4,000080,28.182224,-0.998775
...,...,...,...
195,004540,3.208254,-0.998805
196,004560,14.567581,-0.999055
197,004590,4.969447,-0.998859
198,004650,11.041225,-0.998966


In [ ]:
prediction_result.fillna(0, inplace=True)
prediction_result.sort_values(by='changes', ascending=False, na_position='last', inplace=True)

In [ ]:
prediction_result

,StockCode,final_return,changes
1999,383800,0.000000,0.000000
1740,247660,0.000000,0.000000
1971,352700,0.000000,0.000000
1708,236810,0.000000,0.000000
1943,333620,0.000000,0.000000
...,...,...,...
1181,086520,253.367111,-0.999536
1847,290690,9.985394,-0.999565
278,006740,8.072506,-0.999596
1914,317770,3.288991,-0.999657


In [ ]:
prediction_result.reset_index(drop=True, inplace=True)
prediction_result['순위'] = range(1, 2001)
prediction_result.rename(columns={'StockCode': '종목코드'}, inplace=True)
prediction_result.drop(['final_return', 'changes'], axis=1, inplace=True)
prediction_result


,종목코드,순위
0,383800,1
1,247660,2
2,352700,3
3,236810,4
4,333620,5
...,...,...
1995,086520,1996
1996,290690,1997
1997,006740,1998
1998,317770,1999


In [ ]:
prediction_result['종목코드'] = prediction_result['종목코드'].astype(str).str.zfill(6)
prediction_result['종목코드'] = 'A' + prediction_result['종목코드'].astype(str)

prediction_result.to_csv('2000model_hiddensize100.csv', index=False, encoding='utf-8-sig')



In [ ]:
prediction_result

,종목코드,순위
0,A151910,1
1,A101140,2
2,A214870,3
3,A003560,4
4,A033180,5
...,...,...
1995,A003240,1996
1996,A051910,1997
1997,A006400,1998
1998,A086520,1999
